# Week 4 and 5 deliverables. Data Cleaning and Preparation

In [1]:
import pandas as pd
import numpy as np

sold = pd.read_csv('csv/sold_enriched.csv', low_memory=False)
listings = pd.read_csv('csv/listings_enriched.csv', low_memory=False)

print(f"Sold shape: {sold.shape}")
print(f"Listings shape: {listings.shape}")

Sold shape: (430464, 70)
Listings shape: (591263, 73)


In [2]:
date_cols = ['CloseDate', 'PurchaseContractDate', 'ListingContractDate', 'ContractStatusChangeDate']

for col in date_cols:
    if col in sold.columns:
        sold[col] = pd.to_datetime(sold[col], errors='coerce')
        print(f"Sold — {col}: {sold[col].dtype}")
    if col in listings.columns:
        listings[col] = pd.to_datetime(listings[col], errors='coerce')
        print(f"Listings — {col}: {listings[col].dtype}")

Sold — CloseDate: datetime64[ns]
Listings — CloseDate: datetime64[ns]
Sold — PurchaseContractDate: datetime64[ns]
Listings — PurchaseContractDate: datetime64[ns]
Sold — ListingContractDate: datetime64[ns]
Listings — ListingContractDate: datetime64[ns]
Sold — ContractStatusChangeDate: datetime64[ns]
Listings — ContractStatusChangeDate: datetime64[ns]


In [3]:
numeric_cols = ['ClosePrice', 'ListPrice', 'OriginalListPrice', 'LivingArea',
                'LotSizeAcres', 'BedroomsTotal', 'BathroomsTotalInteger',
                'DaysOnMarket', 'YearBuilt', 'Latitude', 'Longitude']

for col in numeric_cols:
    if col in sold.columns:
        sold[col] = pd.to_numeric(sold[col], errors='coerce')
    if col in listings.columns:
        listings[col] = pd.to_numeric(listings[col], errors='coerce')

print("Numeric dtypes confirmed for sold:")
print(sold[[c for c in numeric_cols if c in sold.columns]].dtypes)

Numeric dtypes confirmed for sold:
ClosePrice               float64
ListPrice                float64
OriginalListPrice        float64
LivingArea               float64
LotSizeAcres             float64
BedroomsTotal            float64
BathroomsTotalInteger    float64
DaysOnMarket               int64
YearBuilt                float64
Latitude                 float64
Longitude                float64
dtype: object


In [4]:
# Sold flags
sold['invalid_close_price'] = sold['ClosePrice'] <= 0
sold['invalid_living_area'] = sold['LivingArea'] <= 0
sold['invalid_days_on_market'] = sold['DaysOnMarket'] < 0
sold['invalid_bedrooms'] = sold['BedroomsTotal'] < 0
sold['invalid_bathrooms'] = sold['BathroomsTotalInteger'] < 0

# Listings flags
listings['invalid_list_price'] = listings['ListPrice'] <= 0
listings['invalid_living_area'] = listings['LivingArea'] <= 0
listings['invalid_days_on_market'] = listings['DaysOnMarket'] < 0
listings['invalid_bedrooms'] = listings['BedroomsTotal'] < 0
listings['invalid_bathrooms'] = listings['BathroomsTotalInteger'] < 0

print("=== Invalid Numeric Value Counts (SOLD) ===")
invalid_cols_sold = ['invalid_close_price', 'invalid_living_area', 'invalid_days_on_market',
                     'invalid_bedrooms', 'invalid_bathrooms']
for col in invalid_cols_sold:
    print(f"  {col}: {sold[col].sum()}")

print("\n=== Invalid Numeric Value Counts (LISTINGS) ===")
invalid_cols_listings = ['invalid_list_price', 'invalid_living_area', 'invalid_days_on_market',
                         'invalid_bedrooms', 'invalid_bathrooms']
for col in invalid_cols_listings:
    print(f"  {col}: {listings[col].sum()}")

=== Invalid Numeric Value Counts (SOLD) ===
  invalid_close_price: 1
  invalid_living_area: 153
  invalid_days_on_market: 49
  invalid_bedrooms: 0
  invalid_bathrooms: 0

=== Invalid Numeric Value Counts (LISTINGS) ===
  invalid_list_price: 0
  invalid_living_area: 389
  invalid_days_on_market: 30
  invalid_bedrooms: 0
  invalid_bathrooms: 0


In [5]:
# listing_after_close_flag: ListingContractDate should come BEFORE CloseDate
sold['listing_after_close_flag'] = sold['ListingContractDate'] > sold['CloseDate']

# purchase_after_close_flag: PurchaseContractDate should come BEFORE CloseDate
sold['purchase_after_close_flag'] = sold['PurchaseContractDate'] > sold['CloseDate']

# negative_timeline_flag: ListingContractDate should come BEFORE PurchaseContractDate
sold['negative_timeline_flag'] = sold['ListingContractDate'] > sold['PurchaseContractDate']

print("=== Date Consistency Flag Counts (SOLD) ===")
print(f"  listing_after_close_flag:   {sold['listing_after_close_flag'].sum()}")
print(f"  purchase_after_close_flag:  {sold['purchase_after_close_flag'].sum()}")
print(f"  negative_timeline_flag:     {sold['negative_timeline_flag'].sum()}")

# Same for listings
listings['listing_after_close_flag'] = listings['ListingContractDate'] > listings['CloseDate']
listings['negative_timeline_flag'] = listings['ListingContractDate'] > listings['PurchaseContractDate']

print("\n=== Date Consistency Flag Counts (LISTINGS) ===")
print(f"  listing_after_close_flag:  {listings['listing_after_close_flag'].sum()}")
print(f"  negative_timeline_flag:    {listings['negative_timeline_flag'].sum()}")

=== Date Consistency Flag Counts (SOLD) ===
  listing_after_close_flag:   65
  purchase_after_close_flag:  241
  negative_timeline_flag:     280

=== Date Consistency Flag Counts (LISTINGS) ===
  listing_after_close_flag:  79
  negative_timeline_flag:    292


In [6]:
# Missing coordinates
sold['missing_coords'] = sold['Latitude'].isnull() | sold['Longitude'].isnull()
listings['missing_coords'] = listings['Latitude'].isnull() | listings['Longitude'].isnull()

# Sentinel null values (0,0)
sold['zero_coords'] = (sold['Latitude'] == 0) | (sold['Longitude'] == 0)
listings['zero_coords'] = (listings['Latitude'] == 0) | (listings['Longitude'] == 0)

# Longitude > 0 (California should be negative longitude)
sold['longitude_positive_flag'] = sold['Longitude'] > 0
listings['longitude_positive_flag'] = listings['Longitude'] > 0

# Out of state / implausible coordinates
# California lat: roughly 32.5 to 42.0, lon: -124.5 to -114.0
sold['out_of_state_flag'] = (
    (sold['Latitude'] < 32.5) | (sold['Latitude'] > 42.0) |
    (sold['Longitude'] < -124.5) | (sold['Longitude'] > -114.0)
) & (~sold['missing_coords'])

listings['out_of_state_flag'] = (
    (listings['Latitude'] < 32.5) | (listings['Latitude'] > 42.0) |
    (listings['Longitude'] < -124.5) | (listings['Longitude'] > -114.0)
) & (~listings['missing_coords'])

print("=== Geographic Data Quality Summary (SOLD) ===")
print(f"  Missing coordinates:      {sold['missing_coords'].sum()}")
print(f"  Zero coordinates:         {sold['zero_coords'].sum()}")
print(f"  Positive longitude:       {sold['longitude_positive_flag'].sum()}")
print(f"  Out of state/implausible: {sold['out_of_state_flag'].sum()}")

print("\n=== Geographic Data Quality Summary (LISTINGS) ===")
print(f"  Missing coordinates:      {listings['missing_coords'].sum()}")
print(f"  Zero coordinates:         {listings['zero_coords'].sum()}")
print(f"  Positive longitude:       {listings['longitude_positive_flag'].sum()}")
print(f"  Out of state/implausible: {listings['out_of_state_flag'].sum()}")

=== Geographic Data Quality Summary (SOLD) ===
  Missing coordinates:      16084
  Zero coordinates:         34
  Positive longitude:       31
  Out of state/implausible: 97

=== Geographic Data Quality Summary (LISTINGS) ===
  Missing coordinates:      80733
  Zero coordinates:         69
  Positive longitude:       79
  Out of state/implausible: 333


In [7]:
sold_before = len(sold)
listings_before = len(listings)

# Create clean filtered versions (remove invalid records)
# We keep flags in the full dataset but save a clean version without invalid records
sold_clean = sold[
    (sold['ClosePrice'] > 0) &
    (sold['LivingArea'] > 0) &
    (sold['DaysOnMarket'] >= 0) &
    (~sold['listing_after_close_flag']) &
    (~sold['purchase_after_close_flag']) &
    (~sold['longitude_positive_flag']) &
    (~sold['out_of_state_flag']) &
    (~sold['missing_coords'])
].copy()

listings_clean = listings[
    (listings['ListPrice'] > 0) &
    (listings['LivingArea'] > 0) &
    (~listings['listing_after_close_flag']) &
    (~listings['longitude_positive_flag']) &
    (~listings['out_of_state_flag']) &
    (~listings['missing_coords'])
].copy()

print("=== Before/After Row Counts ===")
print(f"Sold:     {sold_before} → {len(sold_clean)} rows (removed {sold_before - len(sold_clean)})")
print(f"Listings: {listings_before} → {len(listings_clean)} rows (removed {listings_before - len(listings_clean)})")

# Save
sold_clean.to_csv('sold_clean.csv', index=False)
listings_clean.to_csv('listings_clean.csv', index=False)

print("\nSaved sold_clean.csv and listings_clean.csv")

=== Before/After Row Counts ===
Sold:     430464 → 413598 rows (removed 16866)
Listings: 591263 → 509188 rows (removed 82075)

Saved sold_clean.csv and listings_clean.csv
